In [31]:
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
import string
import time
from datetime import datetime


USERNAME = "gabriel.bezhi@stud.hslu.ch"
PASSWORD = "uBFh6fsx"
BASE_URL = "https://www.zefix.admin.ch/ZefixPublicREST"

auth = HTTPBasicAuth(USERNAME, PASSWORD)
headers = {"Accept": "application/json", "Content-Type": "application/json"}

In [40]:

import time
import requests
import pandas as pd

# BASE_URL, auth, headers werden als bereits definiert angenommen
CANTON = "LU"
SLEEP_BETWEEN = 0.3

# 1-stellige Präfixe: a-z + 0-9 = 36 Kombinationen
prefixes = list(string.ascii_lowercase + string.digits)

all_hits = []

for i, prefix in enumerate(prefixes, 1):
    payload = {
        "name": f"{prefix}*",
        "canton": CANTON,
        "activeOnly": True,
    }

    try:
        resp = requests.post(
            f"{BASE_URL}/api/v1/company/search",
            auth=auth, headers=headers, json=payload, timeout=30,
        )
        resp.raise_for_status()
        search_results = resp.json()
        hits = search_results if isinstance(search_results, list) else search_results.get("list", [])
    except Exception as e:
        print(f"[{i}/{len(prefixes)}] {prefix}*: FEHLER {e}")
        continue

    if hits:
        all_hits.extend(hits)
        print(f"[{i}/{len(prefixes)}] {prefix}*: +{len(hits)} (total: {len(all_hits)})")
    else:
        print(f"[{i}/{len(prefixes)}] {prefix}*: 0 Treffer")

    time.sleep(SLEEP_BETWEEN)

# In DataFrame packen und über uid deduplizieren
df_search = pd.json_normalize(all_hits)
if "uid" in df_search.columns:
    before = len(df_search)
    df_search = df_search.drop_duplicates(subset="uid", keep="first").reset_index(drop=True)
    print(f"\nDedup: {before} → {len(df_search)} unique Firmen")

print(f"Fertig: {len(df_search)} unique Firmen in Kanton {CANTON}")
df_search

[1/36] a*: +2904 (total: 2904)
[2/36] b*: +2686 (total: 5590)
[3/36] c*: +1682 (total: 7272)
[4/36] d*: +1381 (total: 8653)
[5/36] e*: +1386 (total: 10039)
[6/36] f*: +1528 (total: 11567)
[7/36] g*: +1665 (total: 13232)
[8/36] h*: +1707 (total: 14939)
[9/36] i*: +1119 (total: 16058)
[10/36] j*: +564 (total: 16622)
[11/36] k*: +1444 (total: 18066)
[12/36] l*: +1370 (total: 19436)
[13/36] m*: +2584 (total: 22020)
[14/36] n*: +819 (total: 22839)
[15/36] o*: +546 (total: 23385)
[16/36] p*: +1958 (total: 25343)
[17/36] q*: +93 (total: 25436)
[18/36] r*: +1651 (total: 27087)
[19/36] s*: +3932 (total: 31019)
[20/36] t*: +1412 (total: 32431)
[21/36] u*: +270 (total: 32701)
[22/36] v*: +752 (total: 33453)
[23/36] w*: +1177 (total: 34630)
[24/36] x*: +67 (total: 34697)
[25/36] y*: +106 (total: 34803)
[26/36] z*: +437 (total: 35240)
[27/36] 0*: +3 (total: 35243)
[28/36] 1*: +35 (total: 35278)
[29/36] 2*: +30 (total: 35308)
[30/36] 3*: +34 (total: 35342)
[31/36] 4*: +40 (total: 35382)
[32/36] 5*: 

,name,ehraid,uid,chid,legalSeatId,legalSeat,registryOfCommerceId,status,sogcDate,deletionDate,legalForm.id,legalForm.uid,legalForm.name.de,legalForm.name.fr,legalForm.name.it,legalForm.name.en,legalForm.shortName.de,legalForm.shortName.fr,legalForm.shortName.it,legalForm.shortName.en
0,A1 AUTO AG in Liquidation,437580,CHE107251578,CH10030216915,1094,Nottwil,100,BEING_CANCELLED,2026-03-24,None,3,0106,Aktiengesellschaft,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd
1,A1 Automobile GmbH,1434117,CHE338108860,CH10048096209,1093,Neuenkirch,100,ACTIVE,2020-04-27,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
2,A1 engineering ag,1055874,CHE212607256,CH10037944356,1084,Eich,100,ACTIVE,2022-11-01,None,3,0106,Aktiengesellschaft,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd
3,A1 Immobilien & Consulting GmbH,1286247,CHE291691203,CH10048029571,1098,Ruswil,100,ACTIVE,2022-07-13,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
4,A1-Service & Wellness Markus Bünder,1522916,CHE256528001,CH10018136673,1098,Ruswil,100,ACTIVE,2022-02-09,None,1,0101,Einzelunternehmen,Entreprise individuelle,Ditta individuale,Sole proprietorship,EIU,EI,IPI,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35335,8tungnaomi GmbH in Liquidation,1484627,CHE475219866,CH02040738992,1086,Grosswangen,100,BEING_CANCELLED,2025-05-30,None,4,0107,Gesellschaft mit beschränkter Haftung,Société à responsabilité limitée,Società a garanzia limitata,Limited Liability Company,GmbH,Sàrl,Sagl,LLC
35336,95' - Holding AG,1555017,CHE203663296,CH40034517342,1103,Sursee,100,ACTIVE,2025-03-05,None,3,0106,Aktiengesellschaft,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd
35337,969.coffee AG,1308378,CHE313609927,CH10038039590,1024,Emmen,100,ACTIVE,2026-01-15,None,3,0106,Aktiengesellschaft,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd
35338,9.81 Arbeitssicherheit AG,1004437,CHE380242043,CH10037922823,1024,Emmen,100,ACTIVE,2026-02-06,None,3,0106,Aktiengesellschaft,Société anonyme,Società anonima,Corporation,AG,SA,SA,Ltd


In [41]:
def fetch_details(uid: str) -> list[dict]:
    """Holt alle Detail-Einträge zu einer UID. Gibt leere Liste bei Fehler zurück."""
    try:
        r = requests.get(
            f"{BASE_URL}/api/v1/company/uid/{uid}",
            auth=auth, headers={"Accept": "application/json"}, timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        return data if isinstance(data, list) else [data]
    except requests.HTTPError as e:
        print(f"  ! Fehler bei UID {uid}: {e}")
        return []

In [ ]:
all_details = []
for uid in df_search["uid"].dropna().unique():
    print(f"Lade Details für {uid} ...")
    all_details.extend(fetch_details(uid))
    time.sleep(0.2)

df_details = pd.json_normalize(all_details, sep=".")

Lade Details für CHE107251578 ...
Lade Details für CHE338108860 ...
Lade Details für CHE212607256 ...
Lade Details für CHE291691203 ...
Lade Details für CHE256528001 ...
Lade Details für CHE144486401 ...
Lade Details für CHE379851330 ...
Lade Details für CHE334250418 ...
Lade Details für CHE115464225 ...
Lade Details für CHE134913146 ...
Lade Details für CHE175035678 ...
Lade Details für CHE137692125 ...
Lade Details für CHE112841840 ...
Lade Details für CHE104802192 ...
Lade Details für CHE287228842 ...
Lade Details für CHE112157206 ...
Lade Details für CHE303920217 ...
Lade Details für CHE110498317 ...
Lade Details für CHE106835972 ...
Lade Details für CHE495219898 ...
Lade Details für CHE341446292 ...
Lade Details für CHE105310968 ...
Lade Details für CHE106699479 ...
Lade Details für CHE466314595 ...
Lade Details für CHE316021073 ...
Lade Details für CHE108531017 ...
Lade Details für CHE392282368 ...
Lade Details für CHE441768158 ...
Lade Details für CHE439676305 ...
Lade Details f

In [44]:
df_merged = df_search.merge(df_details, on="uid", how="left", suffixes=("_search", "_detail"))

print(f"\nSuche: {len(df_search)} Treffer")
print(f"Details: {len(df_details)} Einträge")
print(f"Merged: {len(df_merged)} Zeilen")

df_merged.head(n=10)


Suche: 35340 Treffer
Details: 35340 Einträge
Merged: 35340 Zeilen


,name_search,ehraid_search,uid,chid_search,legalSeatId_search,legalSeat_search,registryOfCommerceId_search,status_search,sogcDate_search,deletionDate_search,...,address.street,address.houseNumber,address.addon,address.poBox,address.city,address.swissZipCode,zefixDetailWeb.de,zefixDetailWeb.fr,zefixDetailWeb.it,zefixDetailWeb.en
0,A1 AUTO AG in Liquidation,437580,CHE107251578,CH10030216915,1094,Nottwil,100,BEING_CANCELLED,2026-03-24,None,...,Kantonsstrasse,20,,,Nottwil,6207,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
1,A1 Automobile GmbH,1434117,CHE338108860,CH10048096209,1093,Neuenkirch,100,ACTIVE,2020-04-27,None,...,Surseestrasse,14,,,Neuenkirch,6206,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
2,A1 engineering ag,1055874,CHE212607256,CH10037944356,1084,Eich,100,ACTIVE,2022-11-01,None,...,Neumattstrasse,2,,,Eich,6205,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
3,A1 Immobilien & Consulting GmbH,1286247,CHE291691203,CH10048029571,1098,Ruswil,100,ACTIVE,2022-07-13,None,...,Grindel,34,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
4,A1-Service & Wellness Markus Bünder,1522916,CHE256528001,CH10018136673,1098,Ruswil,100,ACTIVE,2022-02-09,None,...,Schwerzistrasse,13,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
5,A1 Steuerberatung GmbH,1520104,CHE144486401,CH10048135462,1098,Ruswil,100,ACTIVE,2022-01-20,None,...,Grindel,34,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
6,A1 Treuhand GmbH,1167251,CHE379851330,CH10047974926,1098,Ruswil,100,ACTIVE,2026-03-13,None,...,Grindel,34,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
7,A1 Vorsorge-/ Vermögensberatung GmbH,1536235,CHE334250418,CH10048142412,1098,Ruswil,100,ACTIVE,2022-05-12,None,...,Grindel,34,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
8,A20-Immobilien AG,973574,CHE115464225,CH14030035675,1059,Kriens,100,ACTIVE,2023-05-31,None,...,Steinhofrain,17,,,Luzern,6005,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...
9,A2 Elektrobau Partner GmbH,1557345,CHE134913146,CH10048151672,1061,Luzern,100,ACTIVE,2022-10-20,None,...,Grossmatte,2b,,,Luzern,6014,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...


In [45]:
df_merged.to_csv("260517_full_zefix_export.csv", index=False, encoding="utf-8-sig")

In [46]:
import re
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime, date
from urllib.parse import urlparse

HEADERS_HRK = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    ),
    "Accept-Language": "de-CH,de;q=0.9",
}

def _parse_eintragsdatum(response_text: str) -> date | None:
    m = re.search(
        r"Eingetragen am</span>\s*<span>(\d{2}\.\d{2}\.\d{4})</span>",
        response_text,
    )
    if not m:
        soup = BeautifulSoup(response_text, "html.parser")
        text = soup.get_text(" ", strip=True)
        m = re.search(
            r"Eingetragen\s+am[\s:]*?(\d{2}\.\d{2}\.\d{4})",
            text,
            flags=re.IGNORECASE,
        )
        if not m:
            return None
    return datetime.strptime(m.group(1), "%d.%m.%Y").date()

def scrape_eintragsdatum_from_url(
    url: str,
    session: requests.Session | None = None,
    timeout: int = 30,
) -> date | None:
    if not isinstance(url, str) or not url:
        return None
    
    sess = session or requests.Session()
    
    # 1) Seite laden → ViewState einsammeln
    r1 = sess.get(url, headers=HEADERS_HRK, timeout=timeout)
    r1.raise_for_status()
    soup = BeautifulSoup(r1.text, "html.parser")
    vs_el = soup.find("input", {"name": "javax.faces.ViewState"})
    if not vs_el:
        return None
    view_state = vs_el.get("value")
    
    # Origin aus der URL ableiten (für den Referer/Origin-Header)
    parsed = urlparse(url)
    origin = f"{parsed.scheme}://{parsed.hostname}"
    
    # 2) AJAX-Postback fürs Lazy-Panel
    ajax_headers = {
        **HEADERS_HRK,
        "Faces-Request": "partial/ajax",
        "X-Requested-With": "XMLHttpRequest",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Origin": origin,
        "Referer": r1.url,
    }
    data = {
        "javax.faces.partial.ajax": "true",
        "javax.faces.source": "idAuszugForm:auszugContentPanel",
        "javax.faces.partial.execute": "idAuszugForm:auszugContentPanel",
        "javax.faces.partial.render": "idAuszugForm:auszugContentPanel",
        "idAuszugForm:auszugContentPanel_load": "true",
        "idAuszugForm": "idAuszugForm",
        "javax.faces.ViewState": view_state,
    }
    r2 = sess.post(url, data=data, headers=ajax_headers, timeout=timeout)
    r2.raise_for_status()
    
    return _parse_eintragsdatum(r2.text)

In [47]:
from tqdm import tqdm

tqdm.pandas()
sess = requests.Session()

def safe_scrape(url):
    try:
        result = scrape_eintragsdatum_from_url(url, session=sess)
        time.sleep(0.2)
        return result
    except Exception as e:
        print(f"Fehler bei {url}: {e}")
        return None

df_merged["eintragsdatum"] = df_merged["cantonalExcerptWeb"].progress_apply(safe_scrape)

/Users/gabriel/.pyenv/versions/3.11.1/lib/python3.11/html/parser.py:170: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  k = self.parse_starttag(i)
 21%|███████▏                          | 7518/35340 [1:18:52<4:36:09,  1.68it/s]

Fehler bei https://lu.chregister.ch/cr-portal/auszug/auszug.xhtml?uid=CHE-486.134.292: 403 Client Error: Forbidden for url: https://lu.chregister.ch/cr-portal/auszug/auszug.xhtml?uid=CHE-486.134.292


 64%|█████████████████████            | 22578/35340 [3:57:14<2:04:40,  1.71it/s]

Fehler bei https://lu.chregister.ch/cr-portal/auszug/auszug.xhtml?uid=CHE-231.270.602: 403 Client Error: Forbidden for url: https://lu.chregister.ch/cr-portal/auszug/auszug.xhtml?uid=CHE-231.270.602


 69%|██████████████████████          | 24432/35340 [4:17:24<29:03:42,  9.59s/it]

Fehler bei https://lu.chregister.ch/cr-portal/auszug/auszug.xhtml?uid=CHE-254.953.534: HTTPSConnectionPool(host='lu.chregister.ch', port=443): Read timed out. (read timeout=30)


 76%|████████████████████████▍       | 26957/35340 [4:49:33<22:21:15,  9.60s/it]

Fehler bei https://lu.chregister.ch/cr-portal/auszug/auszug.xhtml?uid=CHE-345.912.519: HTTPSConnectionPool(host='lu.chregister.ch', port=443): Read timed out. (read timeout=30)


100%|███████████████████████████████████| 35340/35340 [6:22:59<00:00,  1.54it/s]


In [49]:
df_merged

,name_search,ehraid_search,uid,chid_search,legalSeatId_search,legalSeat_search,registryOfCommerceId_search,status_search,sogcDate_search,deletionDate_search,...,address.houseNumber,address.addon,address.poBox,address.city,address.swissZipCode,zefixDetailWeb.de,zefixDetailWeb.fr,zefixDetailWeb.it,zefixDetailWeb.en,eintragsdatum
0,A1 AUTO AG in Liquidation,437580,CHE107251578,CH10030216915,1094,Nottwil,100,BEING_CANCELLED,2026-03-24,None,...,20,,,Nottwil,6207,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,1998-09-07
1,A1 Automobile GmbH,1434117,CHE338108860,CH10048096209,1093,Neuenkirch,100,ACTIVE,2020-04-27,None,...,14,,,Neuenkirch,6206,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2020-04-22
2,A1 engineering ag,1055874,CHE212607256,CH10037944356,1084,Eich,100,ACTIVE,2022-11-01,None,...,2,,,Eich,6205,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2012-04-11
3,A1 Immobilien & Consulting GmbH,1286247,CHE291691203,CH10048029571,1098,Ruswil,100,ACTIVE,2022-07-13,None,...,34,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2016-12-06
4,A1-Service & Wellness Markus Bünder,1522916,CHE256528001,CH10018136673,1098,Ruswil,100,ACTIVE,2022-02-09,None,...,13,,,Ruswil,6017,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2022-02-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35335,8tungnaomi GmbH in Liquidation,1484627,CHE475219866,CH02040738992,1086,Grosswangen,100,BEING_CANCELLED,2025-05-30,None,...,1,,,Grosswangen,6022,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2021-04-26
35336,95' - Holding AG,1555017,CHE203663296,CH40034517342,1103,Sursee,100,ACTIVE,2025-03-05,None,...,24,,,Sursee,6210,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2022-09-28
35337,969.coffee AG,1308378,CHE313609927,CH10038039590,1024,Emmen,100,ACTIVE,2026-01-15,None,...,5,,,Emmenbrücke,6020,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2017-06-08
35338,9.81 Arbeitssicherheit AG,1004437,CHE380242043,CH10037922823,1024,Emmen,100,ACTIVE,2026-02-06,None,...,10,,,Emmenbrücke,6020,https://www.zefix.admin.ch/de/search/entity/li...,https://www.zefix.admin.ch/fr/search/entity/li...,https://www.zefix.admin.ch/it/search/entity/li...,https://www.zefix.admin.ch/en/search/entity/li...,2010-12-23


In [48]:
df_merged.to_csv("260517_full_zefix_export_eintragsdatum.csv", index=False, encoding="utf-8-sig")